# E6 — APTOS2019 optimization

Compare one trained checkpoint and its descendants on the same frozen validation sequence. [Optimization notebook index](README.md) · [HPC execution guide](../../novelty_integrated/optimization/README.md#staged-hpc-campaign) · [Historical backend and pruning audit](../../docs/OPTIMIZATION_COMPATIBILITY.md)

## Original checkpoint and provenance

| Receipt | Value |
| --- | --- |
| Original relative checkpoint | `foundation_e6/APTOS2019/ldam_ordinal_adjacent_seed42_b32_20/run/best.pt` |
| Checkpoint SHA-256 | `f03acdc4fc0325f0c7556d5eafec8a2e91ba8b69d733ddb28d86593f64ea221e` |
| Selected epoch | 15 |
| Frozen manifest SHA-256 | `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` |
| Validation images | 366 |
| Processor SHA-256 | `fd575b890da5a949493e1d1e7a70bfcb9e4b99fe444004d2dbfa253add254741` |
| Recorded architecture | `hf_v1_ordinal` |

The published validation macro-F1 is **0.679204**. It documents checkpoint selection and is separate from new CPU FP32 measurements.
[Published validation evidence](../../results/APTOS2019/e6_ldam_ordinal_adjacent/validation_evidence.json)

## Study design

HF ConvNeXt V1 uses the verified official 224-pixel validation processor. The archived timm ConvNeXtV2 experiments use a different GRN layout and are methodological references. These notebooks exercise the receipt-matched HF V1 adapter.

| Stage | Treatment |
| --- | --- |
| Benchmark original | FP32 eager, TorchScript, ONNX Runtime CPU, OpenVINO CPU. Every export is gated on full-sequence numerical and prediction agreement. |
| Prune without recovery | Independent L1 expansion-channel removals of 10%, 30%, and 50%, rounded down per layer. Report the actual parameter and MAC reduction. |
| Recovery | Optional recovery uses the original experiment's objective, and starts a fresh optimizer for a fixed 1–3 epochs. It prepares each recovered ratio plus the 0% recovery control. `measure-recovered` measures the original, unrecovered and recovered variants together under one CPU allocation. |

Batch 1, full ordered validation by default, 20 warm-ups and three timed passes per backend and scope. CPU affinity and thread count are explicit. Model-only excludes image loading and transforms. End-to-end includes them. Quality uses five grades in order 0–4. FLOPs count two operations per counted Conv2d/Linear MAC and exclude normalization, elementwise operations and I/O. E7 counts shared parameters once and computes work for all three views. The validation split selected the parent checkpoints, so trade-offs are exploratory.

## Execution controls

Default execution dispatches no process. HPC stages require separate allocation arguments. HPC jobs verify the original bytes and write exclusively under the dedicated campaign root. No stage uses final test.

In [ ]:
JOB = 'E6_APTOS2019'
from pathlib import Path
import os
import subprocess
import sys

# Run All is read-only. To dispatch one stage, set it explicitly and supply approved arguments.
RUN_STAGE = None  # audit, prepare, measure, recover, measure-recovered
STAGE_OPTIONS = []  # e.g. allocation arguments from the HPC guide

REPO = Path(os.environ.get("CADVIT_REPOSITORY", Path.cwd())).expanduser().resolve()
REPO = next((p for p in (REPO, *REPO.parents)
             if (p / "novelty_integrated/optimization/hpc.py").is_file()), None)
if REPO is None:
    raise ValueError("Set CADVIT_REPOSITORY to the checkout containing the optimization framework.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

if RUN_STAGE is not None:
    allowed = {"audit", "prepare", "measure", "recover", "measure-recovered"}
    if RUN_STAGE not in allowed:
        raise ValueError("Select a supported stage.")
    subprocess.run([sys.executable, "-m", "novelty_integrated.optimization.hpc", RUN_STAGE,
                    "--job", JOB, *STAGE_OPTIONS], cwd=REPO, check=True)


## Published structured-pruning pilot — before recovery

`pilot_100_v1` used the same ordered 100 frozen validation cases across variants, batch size 1 and untimed FP32 inference on a shared CPU. Grade support in 0–4 order is [54, 9, 28, 4, 5]. Checkpoint epoch 15 came from the original full-validation selection. The original row here uses the pilot subset. The reduction target refers to MLP expansion channels.

| Job | Target | Accuracy | Macro-F1 | Weighted-F1 | QWK | Grade MAE | Δ Macro-F1 | Grade 1 recall | Grade 3 recall |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| E6_APTOS2019 | 0% | 0.7900 | 0.6448 | 0.8043 | 0.8664 | 0.2700 | +0.0000 | 0.7778 | 0.7500 |
| E6_APTOS2019 | 10% | 0.5900 | 0.2544 | 0.5440 | 0.4970 | 0.5800 | -0.3905 | 0.7778 | 0.0000 |
| E6_APTOS2019 | 30% | 0.5600 | 0.2324 | 0.5129 | 0.4581 | 0.6100 | -0.4124 | 0.7778 | 0.0000 |
| E6_APTOS2019 | 50% | 0.5400 | 0.1403 | 0.3787 | 0.0000 | 0.9700 | -0.5046 | 0.0000 | 0.0000 |

The 10% arm changed Macro-F1 by -0.3905. At 50%, the model predicted grade 0 on 100/100 cases. There are only 4 grade-3 cases, so even one case changes that recall substantially.

![Macro-F1 and QWK across pre-recovery pruning ratios for APTOS2019](../../docs/figures/pruning_pilot_aptos2019_quality.png)

### Numerical parity

| Job | Backend vs eager | PASS | Min. argmax agreement | Max. absolute logit difference |
| --- | --- | --- | --- | --- |
| E6_APTOS2019 | TorchScript | 4/4 | 100% | 0 |
| E6_APTOS2019 | ONNX Runtime | 4/4 | 100% | 8.70228e-06 |
| E6_APTOS2019 | OpenVINO | 4/4 | 100% | 4.76837e-06 |

All 12 non-eager comparisons agreed with the corresponding eager argmax. The largest absolute logit difference was 8.70228e-06 under the recorded FP32 absolute and relative tolerances of 0.0001. Agreement checks the export against its own variant.

### Recovery, complexity and CPU timing

Post-recovery evidence is `UNVERIFIED` in this export. Prepared-profile parameter counts and MACs are unavailable. Shared CPU validation recorded no timing, so latency is `NOT_MEASURED`. These data support a descriptive quality comparison on the pilot cases.

[Campaign report](../../docs/PRUNING_PILOT_RESULTS.md) · [Evidence and confusion matrices](../../results/optimization/pilot_100_v1/validation_evidence.json) · [Per-variant parity CSV](../../results/optimization/pilot_100_v1/numerical_parity.csv) · [Python generator](../../docs/pruning_pilot_publication.py)

## Results and trade-offs

| Measurement | Status in this checkout |
| --- | --- |
| Original and pruning benchmark | `NOT_RUN` |
| Post-recovery benchmark | `NOT_RUN` |

Set `CADVIT_OPT_CAMPAIGN` to the external campaign root to display completed provenance-matched results. The display includes latency, throughput, model size, compute, aggregate and per-grade quality, confusion matrices, pruning/recovery deltas and publication figures.

In [ ]:
from IPython.display import display, Markdown, SVG
from novelty_integrated.optimization.notebook import load, render

CAMPAIGN = os.environ.get("CADVIT_OPT_CAMPAIGN")
QUALITY_BACKEND = "eager"  # Detailed per-grade metrics and confusion matrices for this backend.
reports = {}
for stage in ("measurement", "post-recovery"):
    report = load(JOB, CAMPAIGN, stage) if CAMPAIGN else None
    if report is None:
        display(Markdown(f"**{stage}** — NOT_RUN"))
        continue
    if reports and any(report[k] != previous[k] for previous in reports.values()
                       for k in ("protocol_sha256", "sample")):
        raise ValueError("Measurement and post-recovery reports are from different protocols or samples.")
    reports[stage] = report
    display(Markdown(f"### {stage}\n\n" + render(report, QUALITY_BACKEND)))
    for figure in ("comparison.svg", "tradeoff.svg"):
        path = Path(CAMPAIGN).expanduser() / JOB / stage / figure
        display(SVG(filename=str(path)))


## Current conclusion

The unrecovered pruning pilot records substantial quality loss on this sample, despite preserved numerical parity across backends. Selecting a recovered ratio or a CPU latency trade-off requires matched recovery and isolated timing evidence.